# Support Vector Machines – Project Notebook



## 1. Imports & Setup

We import all required libraries. The key ones are:
- **`numpy`** – vectorised linear algebra
- **`scipy.optimize.fmin_l_bfgs_b`** – L-BFGS-B solver (box-constrained quasi-Newton)
- **`matplotlib`** – plotting
- **`sklearn.datasets`** – to load the stand-in IRIS dataset


In [ ]:
import numpy as np
import scipy.optimize
import scipy.special
import sklearn.datasets
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.dpi'] = 120


## 2. Utility Functions

### 2.1 Shape helpers
`vcol` and `vrow` reshape 1-D arrays into column / row vectors.  
This is essential for broadcasting in matrix-vector multiplications.


In [ ]:
def vcol(x):
    """Reshape x into a column vector of shape (n, 1)."""
    return x.reshape((x.size, 1))

def vrow(x):
    """Reshape x into a row vector of shape (1, n)."""
    return x.reshape((1, x.size))


### 2.2 Dataset loading & splitting

`split_db_2to1` splits the dataset **2/3 training – 1/3 validation** using a fixed random seed,  
ensuring reproducibility across all experiments.


In [ ]:
def split_db(D, L, seed=0, ratio=2/3):
    nTrain = int(D.shape[1] * ratio)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    return (D[:, idx[:nTrain]], L[idx[:nTrain]]), (D[:, idx[nTrain:]], L[idx[nTrain:]])


# ── If you have project data, replace this function ─────────────────────────
def load_project_data(path="../../../Project/trainData.txt"):
    D, L = [], []
    with open(path) as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            D.append([float(x) for x in parts[:-1]])
            L.append(int(parts[-1]))
    return np.array(D).T, np.array(L, dtype=np.int32)

# ────────────────────────────────────────────────────────────────────────────

D, L = load_project_data('../../../Project/trainData.txt')

(DTR, LTR), (DVAL, LVAL) = split_db(D, L)
print(f"Training   : {DTR.shape[1]} samples, {DTR.shape[0]} features")
print(f"Validation : {DVAL.shape[1]} samples")
print(f"Class distribution (train) – 0: {(LTR==0).sum()}  1: {(LTR==1).sum()}")


### 2.3 Bayes Risk / DCF Functions  *(from `bayesRisk.py`)*

These functions implement the **Detection Cost Function (DCF)** framework:

| Symbol | Meaning |
|--------|---------|
| $\pi_T$ | Prior probability of the target class |
| $C_{fn}$ | Cost of a false negative (miss) |
| $C_{fp}$ | Cost of a false positive (false alarm) |
| **actDCF** | Actual normalised DCF – uses a fixed threshold $t = -\log\frac{\pi_T C_{fn}}{(1-\pi_T)C_{fp}}$ |
| **minDCF** | Minimum normalised DCF – sweeps all thresholds, measures the best achievable cost |

The optimal Bayes threshold is:
$$t^* = -\log\frac{\pi_T C_{fn}}{(1-\pi_T)C_{fp}}$$

`minDCF` gives a calibration-independent measure of discriminative power;  
`actDCF` reveals whether the scores are well-calibrated for the target application.  
A **large gap** $\text{actDCF} - \text{minDCF}$ means the scores are **poorly calibrated**.


In [ ]:
def compute_confusion_matrix(predictedLabels, classLabels):
    nClasses = int(classLabels.max()) + 1
    M = np.zeros((nClasses, nClasses), dtype=np.int32)
    for i in range(classLabels.size):
        M[predictedLabels[i], classLabels[i]] += 1
    return M

def compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp):
    """Apply optimal Bayes threshold to LLR scores."""
    th = -np.log((prior * Cfn) / ((1 - prior) * Cfp))
    return np.int32(llr > th)

def compute_empirical_Bayes_risk_binary(predictedLabels, classLabels, prior, Cfn, Cfp, normalize=True):
    """Actual (empirical) normalised DCF."""
    M   = compute_confusion_matrix(predictedLabels, classLabels)
    Pfn = M[0, 1] / (M[0, 1] + M[1, 1])   # False Negative Rate
    Pfp = M[1, 0] / (M[0, 0] + M[1, 0])   # False Positive Rate
    bayesError = prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp
    if normalize:
        return bayesError / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    return bayesError

def compute_actDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp, normalize=True):
    pred = compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp)
    return compute_empirical_Bayes_risk_binary(pred, classLabels, prior, Cfn, Cfp, normalize=normalize)

def compute_Pfn_Pfp_allThresholds_fast(llr, classLabels):
    """Sweep all possible thresholds and return (Pfn, Pfp, thresholds)."""
    llrSorter = np.argsort(llr)
    llrSorted = llr[llrSorter]
    clsSorted = classLabels[llrSorter]
    nTrue     = (clsSorted == 1).sum()
    nFalse    = (clsSorted == 0).sum()
    nFN, nFP  = 0, nFalse
    Pfn, Pfp  = [nFN / nTrue], [nFP / nFalse]
    for idx in range(len(llrSorted)):
        if clsSorted[idx] == 1:
            nFN += 1
        else:
            nFP -= 1
        Pfn.append(nFN / nTrue)
        Pfp.append(nFP / nFalse)
    llrSorted = np.concatenate([[-np.inf], llrSorted])
    PfnOut, PfpOut, thOut = [], [], []
    for idx in range(len(llrSorted)):
        if idx == len(llrSorted) - 1 or llrSorted[idx + 1] != llrSorted[idx]:
            PfnOut.append(Pfn[idx])
            PfpOut.append(Pfp[idx])
            thOut.append(llrSorted[idx])
    return np.array(PfnOut), np.array(PfpOut), np.array(thOut)

def compute_minDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp, returnThreshold=False):
    """Minimum DCF over all possible thresholds."""
    Pfn, Pfp, th = compute_Pfn_Pfp_allThresholds_fast(llr, classLabels)
    dcfs = (prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp) / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    idx  = np.argmin(dcfs)
    if returnThreshold:
        return dcfs[idx], th[idx]
    return dcfs[idx]

print("Bayes risk functions loaded ✓")


## 3. SVM Core Functions

### 3.1 Linear SVM – Dual Formulation

**Primal objective** (modified to avoid the bias equality constraint):
$$\hat{J}(\hat{\mathbf{w}}) = \frac{1}{2}\|\hat{\mathbf{w}}\|^2 + C\sum_{i=1}^{n}\max\!\left(0,\; 1 - z_i \hat{\mathbf{w}}^\top\hat{\mathbf{x}}_i\right)$$

where the **extended feature vector** absorbs the bias:
$$\hat{\mathbf{x}}_i = \begin{pmatrix}\mathbf{x}_i \\ K\end{pmatrix}, \quad \hat{\mathbf{w}} = \begin{pmatrix}\mathbf{w} \\ b\end{pmatrix}$$

Setting $K > 1$ weakens regularisation of the bias term (K=1 is the default).

**Dual objective** (negated for minimisation):
$$\hat{L}_D(\boldsymbol{\alpha}) = \frac{1}{2}\boldsymbol{\alpha}^\top \hat{H}\boldsymbol{\alpha} - \boldsymbol{\alpha}^\top \mathbf{1}, \quad 0 \le \alpha_i \le C$$

where $\hat{H}_{ij} = z_i z_j \hat{\mathbf{x}}_i^\top \hat{\mathbf{x}}_j$.

The gradient needed by L-BFGS-B is:
$$\nabla_{\boldsymbol{\alpha}}\hat{L}_D = \hat{H}\boldsymbol{\alpha} - \mathbf{1}$$

After optimisation we recover the primal solution:
$$\hat{\mathbf{w}}^* = \sum_{i=1}^{n}\alpha_i^* z_i \hat{\mathbf{x}}_i$$
and split it as $\mathbf{w}^* = \hat{\mathbf{w}}^*_{1:d}$, $b^* = K \cdot \hat{\mathbf{w}}^*_{d+1}$.

The **duality gap** $\hat{J}(\hat{\mathbf{w}}^*) - \hat{J}_D(\boldsymbol{\alpha}^*)$ should be close to zero at the optimal solution.




In [ ]:
def train_dual_SVM_linear(DTR, LTR, C, K=1):
    """
    Train a linear SVM using the modified dual formulation (bias absorbed into w_hat).

    Parameters
    ----------
    DTR : ndarray (d, n) training features, one sample per column
    LTR : ndarray (n,) binary labels {0, 1}
    C : float regularisation / margin trade-off parameter
    K : float scaling factor for the appended constant feature
        (K=1 → standard bias; K>1 → weaker bias regularisation)

    Returns
    -------
    w : ndarray (d,) weight vector
    b : float bias term
    """
    # Convert labels to +1/-1
    ZTR = LTR * 2.0 - 1.0

    # Append a new row of all k values
    # Instead of separating bias term b, we hide it inside 
    # the weights
    DTR_EXT = np.vstack([DTR, np.ones((1, DTR.shape[1])) * K])

    
    # Computes all Dot Products
    # mutliply by the labels
    H = np.dot(DTR_EXT.T, DTR_EXT) * vcol(ZTR) * vrow(ZTR)

    # Dual SVM Loss Function
    def fOpt(alpha):
        Ha = H @ alpha
        loss = 0.5 * alpha @ Ha - alpha.sum()
        grad = Ha - np.ones(alpha.size)
        return loss, grad

    alphaStar, _, _ = scipy.optimize.fmin_l_bfgs_b(
        fOpt,
        np.zeros(DTR_EXT.shape[1]),
        bounds=[(0, C)] * LTR.size,
        factr=np.nan, pgtol=1e-5
    )

    def primalLoss(w_hat):
        S = (vrow(w_hat) @ DTR_EXT).ravel()
        return 0.5 * np.linalg.norm(w_hat)**2 + C * np.maximum(0, 1 - ZTR * S).sum()

    w_hat = ((alphaStar * ZTR).reshape(1, -1) * DTR_EXT).sum(1)
    w = w_hat[:-1]
    b = w_hat[-1] * K

    pLoss = float(primalLoss(w_hat))
    dLoss = float(-fOpt(alphaStar)[0])
    # Dual Gap Check
    gap = pLoss - dLoss
    print(f" Linear SVM | C={C:.2e}, K={K} | primal={pLoss:.6e}, dual={dLoss:.6e}, gap={gap:.2e}")

    return w, b

print("train_dual_SVM_linear defined ✓")


### 3.2 Kernel SVM

For non-linear separation the SVM dual uses **kernel functions** $k(\mathbf{x}_i, \mathbf{x}_j)$  
instead of explicit dot products $\mathbf{x}_i^\top\mathbf{x}_j$.

The modified kernel matrix is:
$$\hat{H}_{ij} = z_i z_j \left[k(\mathbf{x}_i, \mathbf{x}_j) + \xi\right]$$

The constant $\xi \ge 0$ plays the role of the bias term:  
setting $\xi = 1$ adds a (regularised) bias.

**Classification score** for test sample $\mathbf{x}_t$:
$$s(\mathbf{x}_t) = \sum_{i=1}^{n} \alpha_i^* z_i k(\mathbf{x}_i, \mathbf{x}_t)$$

**Polynomial kernel** of degree $d$ with offset $c$:
$$k(\mathbf{x}_1, \mathbf{x}_2) = (\mathbf{x}_1^\top\mathbf{x}_2 + c)^d$$

**Radial Basis Function (RBF) kernel** with bandwidth $\gamma$:
$$k(\mathbf{x}_1, \mathbf{x}_2) = e^{-\gamma\|\mathbf{x}_1 - \mathbf{x}_2\|^2}$$

The squared distance is computed efficiently as:
$$\|\mathbf{x}_1 - \mathbf{x}_2\|^2 = \|\mathbf{x}_1\|^2 + \|\mathbf{x}_2\|^2 - 2\mathbf{x}_1^\top\mathbf{x}_2$$


In [ ]:
# ── Kernel factory functions ───────────────────────────────────────────────

def polyKernel(degree, c):
    """
    Returns a polynomial kernel function k(D1, D2) = (D1^T D2 + c)^degree.
    D1, D2 are data matrices (d x n1) and (d x n2).
    The result is an (n1 x n2) kernel matrix.
    """
    def polyKernelFunc(D1, D2):
        return (np.dot(D1.T, D2) + c) ** degree
    return polyKernelFunc

def rbfKernel(gamma):
    """
    Returns an RBF kernel function k(D1, D2) = exp(-gamma * ||x1 - x2||^2).
    Uses the identity ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x^T y for efficiency.
    """
    def rbfKernelFunc(D1, D2):
        D1N = (D1**2).sum(0)
        D2N = (D2**2).sum(0)
        Z = vcol(D1N) + vrow(D2N) - 2 * np.dot(D1.T, D2)
        return np.exp(-gamma * Z)
    return rbfKernelFunc

# ── Kernel SVM training ────────────────────────────────────────────────────

def train_dual_SVM_kernel(DTR, LTR, C, kernelFunc, eps=1.0):
    """
    Train a kernel SVM.

    Parameters
    ----------
    DTR : (d, n) training features
    LTR : (n,) labels {0,1}
    C : regularisation parameter
    kernelFunc : callable(D1, D2) -> kernel matrix
    eps : additive constant xi >= 0 (implicit bias term; eps=1 adds bias)

    Returns
    -------
    fScore : callable(DTE) -> scores for test samples DTE (d x m)
    """
    ZTR = LTR * 2.0 - 1.0

    Kmat = kernelFunc(DTR, DTR) + eps
    H = vcol(ZTR) * vrow(ZTR) * Kmat

    def fOpt(alpha):
        Ha = H @ alpha
        loss = 0.5 * alpha @ Ha - alpha.sum()
        grad = Ha - np.ones(alpha.size)
        return loss, grad

    alphaStar, _, _ = scipy.optimize.fmin_l_bfgs_b(
        fOpt,
        np.zeros(DTR.shape[1]),
        bounds=[(0, C)] * LTR.size,
        factr=np.nan, pgtol=1e-5
    )

    def primalLoss(alpha):
        Ha = H @ alpha
        return 0.5 * alpha @ Ha + C * np.maximum(0, 1 - Ha).sum()

    pLoss = float(primalLoss(alphaStar))
    dLoss = float(-fOpt(alphaStar)[0])
    gap = pLoss - dLoss
    print(f" Kernel SVM | C={C:.2e}, eps={eps} | primal={pLoss:.6e}, dual={dLoss:.6e}, gap={gap:.2e}")

    def fScore(DTE):
        Kt = kernelFunc(DTR, DTE) + eps
        return ((alphaStar * ZTR).reshape(-1, 1) * Kt).sum(0)

    return fScore

print("Kernel SVM functions defined ✓")


## 4. Project Task 1 – Linear SVM

### 4.1 Sweep over C (K = 1)

We train the linear SVM for C values on a **logarithmic scale**  
$C \in \{10^{-5}, 10^{-4.5}, \ldots, 10^{0}\}$ (11 values), keeping $K=1$.

For each model we compute:
- **Error rate** at threshold 0
- **minDCF** at $\pi_T = 0.1$  (target application prior)
- **actDCF** at $\pi_T = 0.1$

The target application is $\pi_T = 0.1$, $C_{fn} = C_{fp} = 1$.  
A large actDCF – minDCF gap signals score miscalibration.


In [ ]:
prior_T = 0.1   # application target prior
Cfn, Cfp = 1.0, 1.0

C_values = np.logspace(-5, 0, 11)

results_linear = []

print("Training Linear SVM (K=1) over C values...")
for C in C_values:
    w, b    = train_dual_SVM_linear(DTR, LTR, C, K=1)
    SVAL    = (vrow(w) @ DVAL + b).ravel()
    PVAL    = (SVAL > 0).astype(int)
    err     = (PVAL != LVAL).sum() / float(LVAL.size)
    minDCF  = compute_minDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    actDCF  = compute_actDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    results_linear.append((C, err, minDCF, actDCF))

print("\nDone. Summary:")
print(f"{'C':>12}  {'Error%':>8}  {'minDCF':>8}  {'actDCF':>8}")
for C, err, mDCF, aDCF in results_linear:
    print(f"{C:12.5e}  {err*100:8.2f}  {mDCF:8.4f}  {aDCF:8.4f}")


In [ ]:
# ── Plot minDCF and actDCF vs C ─────────────────────────────────────────────
Cs    = [r[0] for r in results_linear]
mDCFs = [r[2] for r in results_linear]
aDCFs = [r[3] for r in results_linear]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(Cs, mDCFs, 'b-o', label='minDCF  (πT=0.1)', linewidth=2)
ax.semilogx(Cs, aDCFs, 'r--s', label='actDCF  (πT=0.1)', linewidth=2)
ax.set_xlabel('C (regularisation parameter)', fontsize=12)
ax.set_ylabel('Normalised DCF', fontsize=12)
ax.set_title('Linear SVM Effect of C (K=1, non-centered)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('linear_svm_C_sweep.png', dpi=150)
plt.show()
print("Figure saved: linear_svm_C_sweep.png")


### 4.2 Analysis – Linear SVM (non-centered)

**What to observe:**
- **minDCF** measures the best achievable discrimination, regardless of threshold.  
  If minDCF varies significantly across C, regularisation matters.
- **actDCF** uses the theoretically optimal Bayes threshold (assuming scores are LLRs).  
  If actDCF >> minDCF, **the SVM scores are poorly calibrated** and the default threshold is suboptimal.
- For low C (strong regularisation): the model is underfitting → higher minDCF.
- For high C (weak regularisation): the model may overfit, but often actDCF stays high  
  because the score magnitude is not calibrated to the target prior.

**Key insight**: SVM scores have no probabilistic interpretation.  
A score of +3.0 does not mean $P(H_T|x) \approx 1$; the scale is arbitrary.  
Therefore, comparing actDCF with minDCF is essential to assess calibration.


## 5. Project Task 2 – Linear SVM on Centered Data

Mean-centering: we subtract the **training mean** from both training and validation data.  
This removes the DC offset from each feature and can reduce the role of the (regularised) bias.

$$\tilde{\mathbf{x}} = \mathbf{x} - \boldsymbol{\mu}_{\text{train}}, \quad \boldsymbol{\mu}_{\text{train}} = \frac{1}{n}\sum_{i=1}^{n}\mathbf{x}_i$$

> **Important**: the mean is computed **only on the training set** and applied to both sets  
> to avoid data leakage.


In [ ]:
# Center data using training mean only
mu_train = DTR.mean(axis=1, keepdims=True)   # shape (d, 1)
DTR_c    = DTR  - mu_train
DVAL_c   = DVAL - mu_train

print(f"Training mean (first 4 features): {mu_train.ravel()[:4]}")
print("Training Linear SVM (K=1) on CENTERED data...")

results_linear_c = []
for C in C_values:
    w, b   = train_dual_SVM_linear(DTR_c, LTR, C, K=1)
    SVAL   = (vrow(w) @ DVAL_c + b).ravel()
    PVAL   = (SVAL > 0).astype(int)
    err    = (PVAL != LVAL).sum() / float(LVAL.size)
    minDCF = compute_minDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    actDCF = compute_actDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    results_linear_c.append((C, err, minDCF, actDCF))

print("\nDone. Summary (centered):")
print(f"{'C':>12}  {'Error%':>8}  {'minDCF':>8}  {'actDCF':>8}")
for C, err, mDCF, aDCF in results_linear_c:
    print(f"{C:12.5e}  {err*100:8.2f}  {mDCF:8.4f}  {aDCF:8.4f}")


In [ ]:
# ── Side-by-side comparison plot ────────────────────────────────────────────
mDCFs_c  = [r[2] for r in results_linear_c]
aDCFs_c  = [r[3] for r in results_linear_c]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (mD, aD, title) in zip(axes, [
    (mDCFs, aDCFs, 'Non-centered'),
    (mDCFs_c, aDCFs_c, 'Centered')
]):
    ax.semilogx(Cs, mD, 'b-o',  label='minDCF',  linewidth=2)
    ax.semilogx(Cs, aD, 'r--s', label='actDCF',  linewidth=2)
    ax.set_xlabel('C', fontsize=12)
    ax.set_ylabel('Normalised DCF', fontsize=12)
    ax.set_title(f'Linear SVM  {title}', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('linear_svm_centered_comparison.png', dpi=150)
plt.show()
print("Figure saved: linear_svm_centered_comparison.png")


### 5.1 Analysis – Centered vs Non-centered

**What to look for:**
- If centering significantly changes minDCF: the raw features had a large mean shift  
  that was interfering with the margin computation.
- If centering reduces the actDCF-minDCF gap: centering improved calibration (unlikely by itself,  
  but possible if class priors at the origin are more balanced after centering).
- If results are similar: the linear SVM with the extended-feature bias already handled the offset adequately.


## 6. Project Task 3 – Polynomial Kernel SVM (d=2, c=1, ξ=0)

**Polynomial kernel**:
$$k(\mathbf{x}_1, \mathbf{x}_2) = (\mathbf{x}_1^\top\mathbf{x}_2 + 1)^2$$

Setting $c=1$ means the kernel **implicitly accounts for a bias** term (offset feature is always present).  
We therefore set $\xi = 0$ (no additional explicit bias constant in $\hat{H}$).

This kernel corresponds to mapping each sample to a space of **all degree-0, 1, and 2 monomials** of the features.

We use the **original, non-centered features** as specified in the project.


In [ ]:
kfunc_poly2 = polyKernel(degree=2, c=1)

print("Training Polynomial Kernel SVM (d=2, c=1, eps=0) over C values...")
results_poly = []
for C in C_values:
    fScore = train_dual_SVM_kernel(DTR, LTR, C, kfunc_poly2, eps=0.0)
    SVAL   = fScore(DVAL)
    PVAL   = (SVAL > 0).astype(int)
    err    = (PVAL != LVAL).sum() / float(LVAL.size)
    minDCF = compute_minDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    actDCF = compute_actDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    results_poly.append((C, err, minDCF, actDCF))

print("\nDone. Summary (Poly d=2):")
print(f"{'C':>12}  {'Error%':>8}  {'minDCF':>8}  {'actDCF':>8}")
for C, err, mDCF, aDCF in results_poly:
    print(f"{C:12.5e}  {err*100:8.2f}  {mDCF:8.4f}  {aDCF:8.4f}")


In [ ]:
mDCFs_poly = [r[2] for r in results_poly]
aDCFs_poly = [r[3] for r in results_poly]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(Cs, mDCFs_poly, 'b-o',  label='minDCF  (πT=0.1)', linewidth=2)
ax.semilogx(Cs, aDCFs_poly, 'r--s', label='actDCF  (πT=0.1)', linewidth=2)
ax.set_xlabel('C', fontsize=12)
ax.set_ylabel('Normalised DCF', fontsize=12)
ax.set_title('Polynomial Kernel SVM (d=2, c=1, ξ=0)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('poly_svm_C_sweep.png', dpi=150)
plt.show()


### 6.1 Analysis – Polynomial Kernel SVM

**What to observe:**
- The degree-2 polynomial kernel can capture **quadratic decision boundaries**,  
  which may better fit data clusters that are not linearly separable.
- Compare minDCF with the linear SVM: if it is lower, the non-linear boundary helps.
- If actDCF remains high: the non-linear model is also **uncalibrated** – expected for SVMs.
- Compare with logistic regression and MVG models from previous labs:  
  similar minDCF suggests the discriminative information is already captured linearly.


## 7. Project Task 4 – RBF Kernel SVM (Grid Search over γ and C)

The RBF kernel does not implicitly include a bias term (it only measures similarity),  
so we set $\xi = 1$ to add a regularised bias.

**Hyper-parameter grid**:
- $\gamma \in \{e^{-4}, e^{-3}, e^{-2}, e^{-1}\}$
- $C \in \text{logspace}(-3, 2, 11)$

We train all $4 \times 11 = 44$ models and plot minDCF and actDCF  
as a function of C with one line per $\gamma$.


In [ ]:
import itertools

gammas  = [np.exp(-4), np.exp(-3), np.exp(-2), np.exp(-1)]
C_rbf   = np.logspace(-3, 2, 11)

results_rbf = {}   # key: gamma -> list of (C, minDCF, actDCF)

print("Training RBF Kernel SVM (grid search over γ and C)...")
for gamma in gammas:
    results_rbf[gamma] = []
    kfunc = rbfKernel(gamma)
    for C in C_rbf:
        fScore = train_dual_SVM_kernel(DTR, LTR, C, kfunc, eps=1.0)
        SVAL   = fScore(DVAL)
        PVAL   = (SVAL > 0).astype(int)
        err    = (PVAL != LVAL).sum() / float(LVAL.size)
        minDCF = compute_minDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
        actDCF = compute_actDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
        results_rbf[gamma].append((C, err, minDCF, actDCF))
    print(f"  γ={gamma:.4f} done")

print("\nGrid search complete ✓")


In [ ]:
# ── Plot: 4 lines for minDCF and 4 lines for actDCF ────────────────────────
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
gamma_labels = [f'γ=e^{int(round(np.log(g)))}' for g in gammas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric_idx, title in zip(axes, [2, 3], ['minDCF (πT=0.1)', 'actDCF (πT=0.1)']):
    for (gamma, label, color) in zip(gammas, gamma_labels, colors):
        vals = [r[metric_idx] for r in results_rbf[gamma]]
        ax.semilogx(C_rbf, vals, '-o', color=color, label=label, linewidth=2, markersize=5)
    ax.set_xlabel('C', fontsize=12)
    ax.set_ylabel('Normalised DCF', fontsize=12)
    ax.set_title(f'RBF Kernel SVM – {title}', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.savefig('rbf_svm_grid_search.png', dpi=150)
plt.show()
print("Figure saved: rbf_svm_grid_search.png")


Lin LR 0.361

### 7.1 Analysis – RBF Kernel SVM

**What to observe:**
- **Small γ** (e.g., $e^{-4}$): very broad kernel → smooth, nearly-linear boundary → similar to linear SVM.
- **Large γ** (e.g., $e^{-1}$): very narrow kernel → complex, localised boundary → may overfit.
- **Optimal γ and C**: look for the combination that minimises minDCF (best discriminability).
- **Calibration**: the gap between actDCF and minDCF for each configuration.
- **Comparison with previous models**: does the non-linear RBF boundary capture structure  
  that linear models cannot? This depends on whether your data has non-linear class boundaries.

> **Practical note**: if the dataset has been shown (via scatter plots) to contain  
> non-linear clusters, RBF kernels may significantly outperform linear SVMs and logistic regression.


## 8. Optional – Polynomial Kernel d=4, c=1, ξ=0

A degree-4 polynomial kernel captures higher-order interactions between features.

**Motivation** (from the PDF):  
Consider the last 2 features $y_i = x_i^{[4:6]}$.  
A degree-2 mapping sends $y \to z = [y_0 y_1]$ (a 1-D feature).  
With degree-4, additional separating surfaces become available in this transformed space.

$$k(\mathbf{x}_1, \mathbf{x}_2) = (\mathbf{x}_1^\top\mathbf{x}_2 + 1)^4$$


In [ ]:
kfunc_poly4 = polyKernel(degree=4, c=1)

print("Training Polynomial Kernel SVM (d=4, c=1, eps=0) over C values...")
results_poly4 = []
for C in C_values:
    fScore = train_dual_SVM_kernel(DTR, LTR, C, kfunc_poly4, eps=0.0)
    SVAL   = fScore(DVAL)
    PVAL   = (SVAL > 0).astype(int)
    err    = (PVAL != LVAL).sum() / float(LVAL.size)
    minDCF = compute_minDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    actDCF = compute_actDCF_binary_fast(SVAL, LVAL, prior_T, Cfn, Cfp)
    results_poly4.append((C, err, minDCF, actDCF))

print("\nDone. Summary (Poly d=4):")
print(f"{'C':>12}  {'Error%':>8}  {'minDCF':>8}  {'actDCF':>8}")
for C, err, mDCF, aDCF in results_poly4:
    print(f"{C:12.5e}  {err*100:8.2f}  {mDCF:8.4f}  {aDCF:8.4f}")


In [ ]:
mDCFs_poly4 = [r[2] for r in results_poly4]
aDCFs_poly4 = [r[3] for r in results_poly4]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(Cs, mDCFs_poly4, 'b-o',  label='minDCF  (πT=0.1)', linewidth=2)
ax.semilogx(Cs, aDCFs_poly4, 'r--s', label='actDCF  (πT=0.1)', linewidth=2)
ax.set_xlabel('C', fontsize=12)
ax.set_ylabel('Normalised DCF', fontsize=12)
ax.set_title('Polynomial Kernel SVM (d=4, c=1, ξ=0)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('poly4_svm_C_sweep.png', dpi=150)
plt.show()


### 8.1 Analysis – Degree-4 Polynomial Kernel

**Geometric intuition (from the PDF)**:

Consider only the last 2 features: $y_i = x_i^{[4:6]}$.  
The degree-2 map sends $y = [y_0, y_1]^\top \to z = y_0 y_1$ (product feature).

In 1-D feature space $z$:
- **Linear rule** → threshold on $z$ = threshold on the product $y_0 y_1$
- **Quadratic rule** → inequality $z \in [a, b]$ i.e. the product is inside an interval

This matters if the two classes form clusters that are separable in the **product space** but not linearly separable in the original space.

A degree-4 kernel allows richer 1-D separating curves in the product-feature space,  
potentially achieving much lower minDCF if the data has this structure.


## 9. Summary & Comparison

We compare the best models across all configurations.


In [ ]:
def best(results):
    return min(results, key=lambda r: r[2])   # minimum minDCF

b_lin   = best(results_linear)
b_lin_c = best(results_linear_c)
b_poly2 = best(results_poly)
b_poly4 = best(results_poly4)

# Best RBF: search across all gamma/C
best_rbf_row = None
for gamma, rows in results_rbf.items():
    for row in rows:
        if best_rbf_row is None or row[2] < best_rbf_row[2]:
            best_rbf_row = (gamma,) + row

print("=" * 80)
print(f"{'Model':<35} {'Best C':>12}  {'minDCF':>8}  {'actDCF':>8}")
print("-" * 80)
print(f"{'Linear SVM (non-centered)':<35} {b_lin[0]:>12.2e}  {b_lin[2]:>8.4f}  {b_lin[3]:>8.4f}")
print(f"{'Linear SVM (centered)':<35} {b_lin_c[0]:>12.2e}  {b_lin_c[2]:>8.4f}  {b_lin_c[3]:>8.4f}")
print(f"{'Poly SVM (d=2, c=1, ξ=0)':<35} {b_poly2[0]:>12.2e}  {b_poly2[2]:>8.4f}  {b_poly2[3]:>8.4f}")
print(f"{'Poly SVM (d=4, c=1, ξ=0)':<35} {b_poly4[0]:>12.2e}  {b_poly4[2]:>8.4f}  {b_poly4[3]:>8.4f}")
g, C_r, err_r, mD_r, aD_r = best_rbf_row
print(f"{'RBF SVM (γ='+f'{g:.4f}, ξ=1)':<35} {C_r:>12.2e}  {mD_r:>8.4f}  {aD_r:>8.4f}")
print("=" * 80)


## 10. Conclusions

### Key Takeaways

**1. Regularisation (C) matters**  
- Too small C (strong regularisation) → underfitting → higher minDCF.  
- Too large C (weak regularisation) → potential overfitting and slower convergence.  
- The optimal C must be selected via cross-validation (here: single validation split).

**2. SVM scores are not calibrated probabilities**  
- SVM scores are arbitrary real numbers (they do not represent log-likelihood ratios).  
- The gap actDCF − minDCF quantifies calibration loss.  
- In practice, **score calibration** (e.g., logistic regression on top of SVM scores) is required  
  before using Bayes decisions.

**3. Linear SVM vs other linear models**  
- Linear SVM and logistic regression often achieve similar minDCF (both are linear classifiers).  
- The key difference: logistic regression directly optimises cross-entropy (calibrated scores),  
  while SVM optimises margin (geometric criterion, uncalibrated scores).

**4. Centering**  
- Centering can help when the raw feature mean is large relative to the margin.  
- With the modified bias (extended feature with K), centering often has little effect.

**5. Kernel SVMs**  
- The polynomial kernel (d=2) captures second-order interactions.  
- The RBF kernel can model arbitrarily complex boundaries but may overfit for large γ.  
- Better minDCF with kernel SVMs implies non-linear structure in the data.  
- Score calibration is even more important for kernel SVMs (scores can be very different in scale).

### Recommendations for the Report
- Report minDCF as the primary metric (calibration-independent).
- Report actDCF to discuss calibration requirements.
- Compare with MVG, Naive Bayes, and logistic regression models from previous labs.
- Discuss whether the data has non-linear structure based on scatter plots and kernel SVM results.
